# 2.1. Data Visualization - Geo Data

The purpose of this notebook is to conduct data analysis on the geographical locations of the launch sites: Vandenberg AFB Space Launch Complex 4, Cape Canaveral Space Launch Complex 40, Kennedy Space Center Launch Complex 39A. In particular, the following are determined:
- Visualization of the launch sites on the map of the USA and corresponding success rate and orbit type
- Closest highways, coastlines, and railways

In [193]:
import folium
import pandas as pd

from pathlib import Path
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [194]:
file_path = Path.cwd().parent / 'data/processed/api-launch-data-table-class.csv'
spacex_df = pd.read_csv(file_path)
spacex_df.head()

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs,class
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.0,2020-11-16,True,True,1
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.4,2020-11-05,True,True,1
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.0,2020-10-24,True,True,1
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.0,2020-10-18,True,True,1
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.0,2020-10-06,True,True,1


In [195]:
# Color map for class -- green if class==1 else red
def color_map_class(outcome):
    color = 'green' if outcome==1 else 'red'
    return color

spacex_df['color_marker'] = spacex_df['class'].map(color_map_class)
spacex_df

,launch_designator,booster_version,orbit,launch_site,flights,reused,landing_pad,block,serial,longitude,latitude,reused_count,outcome,OrbPay,launch_date,gridfins,legs,class,color_marker
0,2020-084,Falcon 9,Low Earth Orbit,Launch Complex 39A,23,False,JRTI,Block 5,B1061,-80.604282,28.608227,0,True ASDS,13.000,2020-11-16,True,True,1,green
1,2020-078,Falcon 9,Medium Earth Orbit,Space Launch Complex 40,23,False,OCISLY,Block 5,B1062,-80.577357,28.561941,0,True ASDS,4.400,2020-11-05,True,True,1,green
2,2020-074,Falcon 9,Low Earth Orbit,Space Launch Complex 40,20,True,JRTI,Block 5,B1060,-80.577357,28.561941,2,True ASDS,16.000,2020-10-24,True,True,1,green
3,2020-073,Falcon 9,Low Earth Orbit,Launch Complex 39A,14,True,OCISLY,Block 5,B1051,-80.604282,28.608227,5,True ASDS,16.000,2020-10-18,True,True,1,green
4,2020-070,Falcon 9,Low Earth Orbit,Launch Complex 39A,19,True,OCISLY,Block 5,B1058,-80.604282,28.608227,2,True ASDS,16.000,2020-10-06,True,True,1,green
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,2013-010,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0007,-80.577357,28.561941,0,None EXP,9.630,2013-03-01,False,False,0,red
91,2012-054,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0006,-80.577357,28.561941,0,None EXP,8.249,2012-10-08,False,False,0,red
92,2012-027,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0005,-80.577357,28.561941,0,None EXP,7.379,2012-05-22,False,False,0,red
93,2010-066,Falcon 9,Low Earth Orbit,Space Launch Complex 40,1,False,ATL,v1.0,B0004,-80.577357,28.561941,0,False PCL,8.774,2010-12-08,False,False,0,red


In [196]:
# Record of coordinates of each launch site
sites_df = spacex_df.groupby('launch_site')[['latitude', 'longitude']].first()
sites_df

,latitude,longitude
launch_site,,
Launch Complex 39A,28.608227,-80.604282
Space Launch Complex 40,28.561941,-80.577357
Space Launch Complex 4E,34.632000,-120.611000


## Map Initialization and Launch Sites

The purpose of this section is to initialize the map and provide markers on the launch sites; including the outcome of each mission conducted in each site.

In [237]:
# Initialize map
initial_coord = [38.5,-97.5]
site_map = folium.Map(initial_coord, zoom_start=5)

# Add marker cluster
marker_cluster = MarkerCluster()

# Add circles with pop-up and markers over each launch site
for site, coords in sites_df.iterrows():
    coords = coords.to_list()

    # Circle marker -- centered at each site
    circle = folium.Circle(coords, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('{}'.format(site)))

    # Marker showing the name for each site
    site_marker = folium.map.Marker(
        coords,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(-50,-50),
            html='<div style="font-size: 20; color:#d35400;"><b>%s</b></div>' % '{}'.format(site),
            )
        )

    # Marker cluster to show count of missions - expandable to show pins indicating outcome
    for index, record in spacex_df[spacex_df['launch_site']==site][['orbit','color_marker']].iterrows():
        outcome_marker = folium.Marker(
            location = coords,
            icon = folium.Icon(color='white', icon_color=record['color_marker']),
            popup = record['orbit']
        )
        marker_cluster.add_child(outcome_marker)

    site_map.add_child(site_marker)
    site_map.add_child(circle)
    
site_map.add_child(marker_cluster)
site_map

**Findings.** More launches are conducted in the east coast -- SLC 40 or LC 39A -- and the missions are strictly either equatorial orbits with low inclination, or low to high earth orbits. On the other hand, launches conducted in the west coast -- SLC 4E -- are sun synchronous orbits or polar orbits, characterized by high inclination angles. A reason for the large frequency in the east coast is because equatorial orbits like GTOs require a lower amount of energy than that in missions around the polar orbit. To achieve the former, the launch is normally conducted near the equator and directed eastward so that the surface of the earth contributes to an optimal degree to the final speed of the launch. Given the direction and inclination angle of polar orbits, such missions do not get to have the 'free ride' that the Earth's rotation provides; thus requiring more energy.

## Closest highways, railways, and coastlines

Here I plotted markers to indicate the distance of each launch site to the nearest map layers -- transportation infrastructures such as highways and railways, or physical locations such as coast lines.

I first converted the launch sites DataFrame into a geopandas GeoDataFrame, containing the site names and their corresponding longitude and latitude stored together in a shapely `Points` object.

I then created a function that plots the line segments given the data for map layers. This automates distance calculations and plotting on the site map, and all that is needed to be done is to find data for each map layers -- The ones considered are highways, railways, and coastlines. 

In [198]:
from shapely.ops import nearest_points
from shapely import LineString
from shapely.geometry import Point
import topojson as tp
import geopandas as gpd

# GeoPandas DataFrame for Launch Sites
sites_gdf = gpd.GeoDataFrame(
    sites_df, 
    geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude),
    crs='EPSG:4326'
)

def nearest_to_site(sites_gdf: gpd.GeoDataFrame, layers_gdf: gpd.GeoDataFrame, site_map: folium.Map, layer_type: None|str = None, color: str = 'grey') -> None:
    """
    Function to plot line segments to mark distances of launch sites to the
    nearest map layers (e.g., highways, railways, coastlines)

    :param sites_gdf: gpd.GeoDataFrame, GeoDataFrame of launch sites with their corresponding geometries
    
    :param layers_gdf: gpd.GeoDataFrame, GeoDataFrame of map layers with their corresponding geometries
    
    :param site_map: folium.Map, Folium map on which the distance markers are plotted.
    
    :param layer_type: None or str; type of layer -- transportation infrastructures like 'highways' or 'railways', 
                       or physical layers like 'coastlines'. Pass the string to be used as the name of the layer
                       type or pass the column name (as str) in the layers_gdf to be used as the name of the layer.
                       If None is passed, function uses 'layer' by default.

    :param color: str; color for the segment (blue, purple, black, etc.)
                       
    :return: None; the function adds the markers directly to `site_map`.
    """
    
    # Reproject
    sites_proj = sites_gdf.to_crs(epsg=5070)
    layers_proj = layers_gdf.to_crs(epsg=5070)
    
    # Calculate distance
    # Geopandas uses shapely under the hood to calculate the distance
    # Calculated distance is the minimum distance from the point (launch site) to the line (highway)
    for site_name, record in sites_proj[['geometry']].iterrows():
        site_geom = record.values[-1]
    
        #Calculate distance
        distance = layers_proj.geometry.distance(site_geom)
        arg_nearest = distance.idxmin()
        nearest_distance = distance.min()/1000
    
        #Define distance segment
        _, nearest_pt = nearest_points(site_geom, layers_proj.loc[arg_nearest, 'geometry'])
        line_segment = LineString([site_geom, nearest_pt])

        #Calculate Midpoint
        midpoint = gpd.GeoSeries(
            data = line_segment.interpolate(0.5, normalized=True),
            crs = 5070
        ).to_crs(epsg=4326).values[-1]
        
        # Plot marker for nearest layer
        popup = folium.GeoJsonPopup(
            fields = ['description', 'distance'],
            aliases = ['', ''],
            localize = True,
            labels = True,
        )

        if not layer_type:
            layer_type = 'layer'
        else:
            if layer_type in layers_gdf.columns:
                layer_type = layers_gdf.loc[arg_nearest, layer_type]

        markup = f"""
            <a>
                <div style="font-size: 0.8em;">
                <div style="width: 10px;
                            height: 10px;
                            border: 1px solid black;
                            border-radius: 5px;
                            background-color: orange;">
                </div>
            </div>
            </a>
        """
        
        folium.GeoJson(
            gpd.GeoDataFrame(data = dict(geometry = [nearest_pt], 
                                         description = ['Nearest {} to {}'.format(layer_type, site_name)],
                                         distance = ['{:.3f} km'.format(nearest_distance)]),
                            crs=5070).to_crs(epsg=4326),
            popup = popup,
            marker = folium.Marker(icon=folium.DivIcon(html = markup)),
        ).add_to(site_map)
        
        # Plot line segment
        folium.GeoJson(
            gpd.GeoDataFrame(data = dict(geometry = [line_segment]),
                             crs = 5070).to_crs(epsg=4326),
            style_function = lambda x: {'color': color, 'dashArray': "5,10"}
        ).add_to(site_map)

        # Add line marker
        folium.map.Marker(
            [midpoint.y, midpoint.x],
            icon = DivIcon(
                icon_size=(100,100),
                icon_anchor=(0,0),
                html = '<div style="font-size: 14; color:{};"><b>{:.3f} km</b></div>'.format(color, nearest_distance)
            )
        ).add_to(site_map)

### Data for Highways

In [199]:
import requests
import json

us_roadmap_topo = requests.get(
    'https://gist.githubusercontent.com/bricedev/96d2113bd29f60780223/raw/957d51ac88a6de442cf73b9efa8615fce9f9577e/usroads.json'
).json()

file_path = Path.cwd().parent / 'data' / 'raw' / 'us_road_map.json'
with open(file_path, 'w') as f:
    json.dump(us_roadmap_topo, f)

In [200]:
# GeoPandas Dataframe for US Highways -- Extract major highways
roadmap_topo = tp.Topology(us_roadmap_topo, object_name="roads")
roads_gdf = roadmap_topo.to_gdf(crs="EPSG:4326")
roads_gdf = roads_gdf[roads_gdf['type']=='Major Highway'].reset_index(drop=True)

In [238]:
nearest_to_site(sites_gdf, roads_gdf, site_map, layer_type='type', color='purple')

In [239]:
site_map

### Data for Railways

In [203]:
railway_geojson = Path.cwd().parent / 'data/external/us_railways.geojson'
railway_gdf = gpd.read_file(railway_geojson)
railway_gdf = railway_gdf[railway_gdf['NET'] == 'M'][['geometry']]

In [240]:
nearest_to_site(sites_gdf, railway_gdf, site_map, layer_type='Major Railway', color='#800020')

In [241]:
site_map

### Data for Coastline

#### West Coast

In [208]:
# Launch sites at the west coast
sites_gdf_west = sites_gdf.iloc[2:,:]

# Coast line gdf from geodata that is accurate at the west coast
coastline_path = Path.cwd().parent / "data/external/ne_coastline.zip"
zip_uri = f"zip://{coastline_path.as_posix()}"
west_coastline_gdf = gpd.read_file(zip_uri)

In [209]:
from tqdm import tqdm

ne_coastline_url = 'https://naciscdn.org/naturalearth/10m/physical/ne_10m_coastline.zip'
ne_coastline_path = Path.cwd().parent / 'data/external/ne_coastline.zip'
with requests.get(ne_coastline_url, stream=True) as r:
    r.raise_for_status()
    with open(ne_coastline_path, 'wb') as f:
        for chunk in tqdm(r.iter_content(chunk_size=8192), desc='Downloading coastline data from naturalearthdata.com'):
            f.write(chunk)

zip_uri_1 = f"zip://{ne_coastline_path.as_posix()}"
west_coastline_gdf_1 = gpd.read_file(zip_uri_1)
west_coastline_gdf.equals(west_coastline_gdf_1)

True

In [242]:
nearest_to_site(sites_gdf_west, west_coastline_gdf, site_map, layer_type='featurecla', color='blue')

In [243]:
site_map

#### East Coast

In [213]:
# Launch sites at the east coast
sites_gdf_east = sites_gdf.iloc[0:2,:]

# From geodata that is accurate on the east coast
florida_gdf = gpd.read_file(Path.cwd().parent / 'data/external/florida_coastline.geojson')
florida_gdf['geometry'] = florida_gdf.geometry.boundary
sites_gdf = sites_gdf.iloc[1:,:]

In [231]:
nearest_to_site(sites_gdf_east, florida_gdf, site_map, layer_type = 'Coastline', color='blue')

In [232]:
site_map

The distance calculations for each launch site to the nearest highway, railway, and coastline were added in the processed dataset. These distances were considered as features instead of the corresponding name of the launch sites and coordinates. These distances test a specific hypothesis (proximity to coast/highway/rail) and can generalize to new locations, while `launch_site` is just a label that can only match sites already seen, and raw coordinates (`longitude` and `latitude`) don't directly capture the geographic mechanism that matters.

Therefore, a script was added in `dataset.py` to add columns for distance to nearest highway, railway, and coastline. 

In [216]:
from shapely.ops import nearest_points
from shapely import LineString
from shapely.geometry import Point
import topojson as tp
import geopandas as gpd
from tqdm import tqdm

def download_layers_data(folder_path: None|str|Path = None) -> dict[str, Path]:
    """
    Function for downloading geodata for highways, railways, and coastlines

    :param folder_path: None, str, or Path; default None; folder path where geodata are cached
                        if None, the files are saved in the default path:
                        parent directory > data/external

    :return file_paths: dict[str, Path]; Dictionary of file paths.
    """
    if not folder_path:
        folder_path = Path.cwd().parent / 'data/external'
    else:
        folder_path = Path(folder_path)

    folder_path.mkdir(parents=True, exist_ok=True)

    # File Paths
    file_paths = {
        'US roadmap': folder_path / 'us_road_map.json',
        'US railways': folder_path / 'us_railways.geojson',
        'Global coastline': folder_path / 'ne_coastline.zip',
        'Florida coastline': folder_path / 'florida_coastline.geojson'
    }

    # URLs
    urls = [
        # US Roads
        'https://gist.githubusercontent.com/bricedev/96d2113bd29f60780223/raw/957d51ac88a6de442cf73b9efa8615fce9f9577e/usroads.json',
        
        # US Railways
        '/'.join([ 
                'https://services.arcgis.com',
                'xOi1kZaI0eWDREZv',
                'arcgis',
                'rest',
                'services',
                'NTAD_North_American_Rail_Network_Lines',
                'FeatureServer',
                'replicafilescache',
                'NTAD_North_American_Rail_Network_Lines_-5214657740406327753.geojson'
            ]), 

        # Global Coastline Data - from Natural Earth
        'https://naciscdn.org/naturalearth/10m/physical/ne_10m_coastline.zip',

        # Florida Coastline Data - from ArcGIS
        'https://hub.arcgis.com/api/v3/datasets/eda0c60e98cd43af9422dc5ea54d8d56_2/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1'
    ]

    for key, file, url in zip(file_paths.keys(), file_paths.values(), urls):
        if not file.is_file():
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                total = int(r.headers.get('Content-Length', 0))
                with open(file, 'wb') as f:
                    for chunk in tqdm(r.iter_content(chunk_size=8192), total = total // 8192, unit='chunk', desc=f'Downloading {key} geodata'):
                        f.write(chunk)

    return file_paths

def add_nearest_highway(data: pd.DataFrame, us_roadmap: None|str|Path = None) -> pd.DataFrame:
    """
    Function to calculate distance of corresponding launch site to nearest highway.
    Adds the distance to nearest highway column to `data`.

    :param data: pd.DataFrame; launch data
    
    :param us_roadmap: None, or str, or Path; Default None; file path to US roadmap geodata. If None,
                       function loads the geodata from the default path:
                       parent directory > data/external/us_road_map.json
                       
    :return data: pd.DataFrame; launch data with `nearest_highway` column
    """
    # Load US Roadmap geodata
    if not us_roadmap:
        us_roadmap = Path.cwd().parent / 'data/external/us_road_map.json'

    with open(us_roadmap, 'rb') as f:
        roads_json = json.load(f)
        
    roads_topo = tp.Topology(roads_json, object_name="roads")
    roads_gdf = roads_topo.to_gdf(crs="EPSG:4326")
    mask = roads_gdf['type']=='Major Highway'
    roads_gdf = roads_gdf[mask].reset_index(drop=True).to_crs(epsg=5070) #reprojected to EPSG:5070

    # Define function -- convert reproject (lon,lat) to EPSG:5070 and calculate nearest distance
    def func(row):
        lon = row.longitude
        lat = row.latitude
        row_conv = gpd.GeoSeries([Point(lon,lat)], crs="EPSG:4326").to_crs(epsg=5070)[0]
        return roads_gdf.geometry.distance(row_conv).min()/1000

    # Apply function to data
    data['nearest_highway'] = data.apply(func, axis=1)
    
    return data

def add_nearest_railway(data: pd.DataFrame, us_railways: None|str|Path = None) -> pd.DataFrame:
    """
    Function to calculate distance of corresponding launch site to nearest railway.
    Adds the distance to nearest railway column to `data`.

    :param data: pd.DataFrame; launch data
    
    :param us_railways: None, or str, or Path; Default None; file path to US railways geodata. If None,
                        functions loads the geodata from the default path:
                        parent directory > data/external/us_railways.geojson
                        
    :return data: pd.DataFrame; launch data with `nearest_railway` column
    """
    # Load US Roadmap geodata
    if not us_railways:
        us_railways = Path.cwd().parent / 'data/external/us_railways.geojson'

    railways_gdf = gpd.read_file(us_railways)
    mask = railways_gdf['NET'] == 'M'
    railways_gdf = railways_gdf[mask][['geometry']].to_crs(epsg=5070) #reprojected to EPSG:5070

    # Define function -- convert reproject (lon,lat) to EPSG:5070 and calculate nearest distance
    def func(row):
        lon = row.longitude
        lat = row.latitude
        row_conv = gpd.GeoSeries([Point(lon,lat)], crs="EPSG:4326").to_crs(epsg=5070)[0]
        return railways_gdf.geometry.distance(row_conv).min()/1000

    # Apply function to data
    data['nearest_railway'] = data.apply(func, axis=1)
    
    return data

def add_nearest_coastline(data: pd.DataFrame, global_coastline: None|str|Path = None, florida_coastline: None|str|Path = None) -> pd.DataFrame:
    """
    Function to calculate distance of corresponding launch site to nearest coastline.
    Adds the distance to nearest coastline column to `data`
    
    For accuracy purposes, the Florida coastline geodata will be used for launch sites based in Florida;
    otherwise, the global coastline geodata will be used.

    :param data: pd.DataFrame; launch data
    
    :param global_coastline: None, or str, or Path; Default None; file path to global coastline geodata. If None,
                             function loads geodata from the default path:
                             parent directory > data/external/ne_coastline.zip

    :param florida_coastline: None, or str, or Path; Default None; file path to Florida coastline geodata. If None,
                              function loads geodata from the default path:
                              parent directory > data/external/florida_coastline.geojson

    :return data: pd.DataFrame; launch data with `nearest_coastline` column
    """

    # Load global coastline data
    if not global_coastline:
        global_coastline = Path.cwd().parent / 'data/external/ne_coastline.zip'

    zip_uri = f"zip://{global_coastline.as_posix()}"
    global_coastline_gdf = gpd.read_file(zip_uri).to_crs(epsg=5070)

    # Load Florida coastline data
    if not florida_coastline:
        florida_coastline = Path.cwd().parent / 'data/external/florida_coastline.geojson'
    
    florida_gdf = gpd.read_file(florida_coastline)
    florida_gdf['geometry'] = florida_gdf.geometry.boundary
    florida_gdf = florida_gdf.to_crs(epsg=5070)

    # Define function -- convert reproject (lon,lat) to EPSG:5070 and calculate nearest distance
    def func(row):
        lon = row.longitude
        lat = row.latitude
        row_conv = gpd.GeoSeries([Point(lon,lat)], crs="EPSG:4326").to_crs(epsg=5070)[0]

        if lon >= -87 and lat <= 31: # Approximate range for places in Florida
            return florida_gdf.geometry.distance(row_conv).min()/1000

        else:
            return global_coastline_gdf.geometry.distance(row_conv).min()/1000

    # Apply function to data
    data['nearest_coastline'] = data.apply(func, axis=1)
    
    return data

def add_nearest(csv_path: str|Path, save_path:None|str|Path = None) -> Path:
    """
    Utility function to run `add_nearest_highway`, `add_nearest_railway`, and `add_nearest_coastline`
    functions on the csv file containing the dataset.

    :param csv_path: str or Path; file path where csv file containing launch data is saved
    :param save_path: None, str, or Path; default None; file path where the processed csv file
                      is saved. If None, the csv file is saved in the current working directory.
    :return save_path: as Path
    """

    try:
        df = pd.read_csv(csv_path)
        
    except FileNotFoundError as e:
        print('Please check if file exists: {}'.format(e))

    if not save_path:
        save_path = Path.cwd()
    
    add_nearest_highway(df)
    add_nearest_railway(df)
    add_nearest_coastline(df)

    df.to_csv(save_path, index=False)

    return save_path

In [219]:
add_nearest_highway(spacex_df)
spacex_df[['longitude', 'latitude', 'nearest_highway']].value_counts()

longitude    latitude   nearest_highway
-80.577357   28.561941  17.694811          57
-80.604282   28.608227  22.499530          23
-120.611000  34.632000  30.520335          15
Name: count, dtype: int64

In [220]:
add_nearest_railway(spacex_df)
spacex_df[['longitude', 'latitude', 'nearest_railway']].value_counts()

longitude    latitude   nearest_railway
-80.577357   28.561941  21.169013          57
-80.604282   28.608227  19.984845          23
-120.611000  34.632000  1.252787           15
Name: count, dtype: int64

In [221]:
add_nearest_coastline(spacex_df)
spacex_df[['longitude', 'latitude', 'nearest_coastline']].value_counts()

longitude    latitude   nearest_coastline
-80.577357   28.561941  0.930777             57
-80.604282   28.608227  0.544885             23
-120.611000  34.632000  1.468531             15
Name: count, dtype: int64